In [31]:
import shutil
import os

# Define source dataset folder (where the current dataset is stored)
source_folder = r"C:/Users/abina/Downloads/archive (1)"  # Modify this to your dataset location

# Define destination folder correctly
destination_folder = r"C:/Users/abina/DietMonitoringSystem/dataset"

# Ensure the destination folder exists
os.makedirs(destination_folder, exist_ok=True)


In [33]:
import os
source_folder = r"C:/Users/abina/Downloads/archive (1)"
print("Folders found:", os.listdir(source_folder))


Folders found: ['evaluation', 'training', 'validation']


In [35]:
import shutil
import os

# Define paths
source_folder = r"C:/Users/abina/Downloads/archive (1)"
destination_folder = r"C:/Users/abina/DietMonitoringSystem/dataset"

# Ensure destination folder exists
os.makedirs(destination_folder, exist_ok=True)

# Get all folders inside the source dataset
folders = os.listdir(source_folder)

print(f"📂 Source Folder Contents: {folders}")  # Debugging step

# Move each folder into the project's dataset
for folder in folders:
    source_path = os.path.join(source_folder, folder)
    destination_path = os.path.join(destination_folder, folder)

    # Ensure it's a valid folder before moving
    if os.path.isdir(source_path):
        print(f"🔄 Moving {source_path} → {destination_path}")  # Debugging step
        shutil.move(source_path, destination_path)
    else:
        print(f"⚠️ Skipped (Not a folder): {source_path}")  # Debugging step

# Final verification
print("📂 Folders in Project Dataset:", os.listdir(destination_folder))


📂 Source Folder Contents: ['evaluation', 'training', 'validation']
🔄 Moving C:/Users/abina/Downloads/archive (1)\evaluation → C:/Users/abina/DietMonitoringSystem/dataset\evaluation
🔄 Moving C:/Users/abina/Downloads/archive (1)\training → C:/Users/abina/DietMonitoringSystem/dataset\training
🔄 Moving C:/Users/abina/Downloads/archive (1)\validation → C:/Users/abina/DietMonitoringSystem/dataset\validation
📂 Folders in Project Dataset: ['evaluation', 'training', 'validation']


In [37]:
print("Folders in Project Dataset:", os.listdir(destination_folder))


Folders in Project Dataset: ['evaluation', 'training', 'validation']


In [45]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader
import os
import numpy as np

# Load ResNet-18 for feature extraction
resnet18 = models.resnet18(pretrained=True)
resnet18.fc = torch.nn.Identity()  # Remove the classification layer to get feature embeddings
resnet18.eval()  # Set model to evaluation mode

# Define dataset paths using absolute paths
dataset_path = r"C:/Users/abina/DietMonitoringSystem/dataset"
train_dir = os.path.join(dataset_path, "training")
val_dir = os.path.join(dataset_path, "validation")

# Define image transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),  
    transforms.ToTensor(),          
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  
])

# Load dataset
train_data = datasets.ImageFolder(train_dir, transform=transform)
val_data = datasets.ImageFolder(val_dir, transform=transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)

# Debugging print statements to verify dataset loading
print(f"✅ Training directory: {train_dir}")
print(f"✅ Validation directory: {val_dir}")
print(f"📂 Total Classes: {len(train_data.classes)}")
print("📂 Class Names:", train_data.classes)



✅ Training directory: C:/Users/abina/DietMonitoringSystem/dataset\training
✅ Validation directory: C:/Users/abina/DietMonitoringSystem/dataset\validation
📂 Total Classes: 11
📂 Class Names: ['Bread', 'Dairy product', 'Dessert', 'Egg', 'Fried food', 'Meat', 'Noodles-Pasta', 'Rice', 'Seafood', 'Soup', 'Vegetable-Fruit']


In [123]:
def extract_features(model, loader):
    features = []
    labels = []
    
    with torch.no_grad():
        for images, targets in loader:
            outputs = model(images)  
            features.append(outputs.view(outputs.shape[0], -1).numpy())  
            labels.extend(targets.numpy())  

    return np.vstack(features).astype(np.float32), np.array(labels).astype(np.int64)


# Extract features for training & validation
X_train, y_train = extract_features(resnet18, train_loader)
X_test, y_test = extract_features(resnet18, val_loader)

print("Feature extraction complete!")


Feature extraction complete!


In [127]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Train SVM model
svm = SVC(kernel="linear")  # You can change "linear" to "rbf" for a nonlinear classifier
svm.fit(X_train, y_train)

# Make predictions on validation data
y_pred = svm.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

print(f"✅ Model Training Complete!")
print(f"📈 SVM Accuracy on Validation Set: {accuracy * 100:.2f}%")



✅ Model Training Complete!
📈 SVM Accuracy on Validation Set: 78.43%


In [129]:
import pickle
import os

# Define the model save path
model_folder = r"C:/Users/abina/DietMonitoringSystem/models"
os.makedirs(model_folder, exist_ok=True)  # Ensure the folder exists

model_path = os.path.join(model_folder, "svm_model.pkl")

# Save the trained SVM model
with open(model_path, "wb") as f:
    pickle.dump(svm, f)

print(f"✅ SVM model saved at: {model_path}")



✅ SVM model saved at: C:/Users/abina/DietMonitoringSystem/models\svm_model.pkl


In [131]:
print("Model Exists:", os.path.exists(model_path))


Model Exists: True


In [4]:
import os

model_folder = "models"
os.makedirs(model_folder, exist_ok=True)  # Ensure the folder exists

model_path = os.path.join(model_folder, "svm_model.pkl")

print(f"✅ Model path set: {model_path}")


✅ Model path set: models\svm_model.pkl


In [9]:
food_labels = {
    0: "Bread",
    1: "Dairy Product",
    2: "Dessert",
    3: "Egg",
    4: "Fried Food",
    5: "Meat",
    6: "Noodles-Pasta",
    7: "Rice",
    8: "Seafood",
    9: "Soup",
    10: "Vegetable-Fruit"
}


In [10]:
import numpy as np
import torch
import pickle
import torchvision.transforms as transforms
from PIL import Image

# Load the saved SVM model
with open(model_path, "rb") as f:
    svm_model = pickle.load(f)

# Load ResNet for feature extraction
resnet18 = torch.hub.load("pytorch/vision:v0.10.0", "resnet18", pretrained=True)
resnet18.fc = torch.nn.Identity()
resnet18.eval()

# Image transformation pipeline (matches Flask processing)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def extract_features(image):
    """Extract features from an image using ResNet18."""
    image = transform(image).unsqueeze(0)
    with torch.no_grad():
        features = resnet18(image).flatten().numpy()
    return features

# Test with an image
image_path = "C:/Users/abina/Downloads/bread.jpg"  # Replace with actual image path
image = Image.open(image_path)

# Extract features and classify
features = extract_features(image)
prediction = svm_model.predict([features])[0]
food_label = food_labels.get(prediction, "Unknown Food")

print(f"✅ Predicted food category: {food_label} (Category {prediction})")


Using cache found in C:\Users\abina/.cache\torch\hub\pytorch_vision_v0.10.0
C:\Users\abina\anaconda\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\abina\anaconda\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


✅ Predicted food category: Bread (Category 0)
